In [ ]:
import numpy as np 
import pandas as pd 
import os
import pydicom
from glob import glob
from collections import defaultdict

DATA_PATH = '/kaggle/input/competitions/rsna-knee-abnormality-detection'

class DICOMExtractor :
    def __init__(self, data_path) :
        self.data_path = data_path

    def _getStudyInstanceUID(self, file = 'train.csv') -> list[str] :
        df = pd.read_csv(os.path.join(self.data_path, file))
        return df['StudyInstanceUID'].to_list()

    def _getSeriesInstanceUID(self, file = 'train_series.csv') -> dict:
        df = pd.read_csv(os.path.join(self.data_path, file))
        seriesInstanceUID = defaultdict(list)
        for study, series in zip(df['StudyInstanceUID'], df['SeriesInstanceUID']) :
            seriesInstanceUID[study].append(series)

        return seriesInstanceUID

    
    def getDICOM(self, metadata_only = False, file = 'train_series'):
        dicomInstances = {}
        for study, series in self._getSeriesInstanceUID().items():
            for ser in series:
                series_dir = os.path.join(self.data_path, file, study, ser)
                paths = sorted(glob(os.path.join(series_dir, "*.dcm")))
                
                if not paths:
                    continue
    
                datasets = []
                for p in paths:
                    try:
                        datasets.append(pydicom.dcmread(p, stop_before_pixels=metadata_only))
                    except Exception as e:
                        print(f"skipping {p}: {type(e).__name__}: {e}")
    
                if not datasets:
                    continue
    
                iop = np.array(datasets[0].ImageOrientationPatient, float)
                normal = np.cross(iop[:3], iop[3:])
                datasets.sort(key=lambda d: float(np.dot(np.array(d.ImagePositionPatient, float), normal)))
                yield (study, ser), datasets
        


    
if __name__ == '__main__' :
    d = DICOMExtractor(DATA_PATH)
    print(d)
    print(next(d.getDICOM()))
    

In [6]:
# -*- coding: utf-8 -*-
"""Rule-based label extraction from multilingual radiology reports.

Turns a free-text knee MRI report, in any of ~12 languages, into 12 soft
labels in [0, 1] suitable for training a vision model.

    labeler = ClinicalNoteLabeler()
    labeler.to_soft_labels("No ACL tear. Medial meniscus posterior horn tear.")
    # {'ACL': 0.02, 'medial_meniscus': 0.95, ...}

Stdlib only. No model, no network, no GPU.

DESIGN
------
Six small pieces rather than one big class, so each can be tested and swapped
independently:

    Certainty          the six-level ordinal scale and its mapping to numbers
    Mention            one occurrence of one finding in one text unit
    LabelResult        the final per-label verdict, with provenance
    Vocabulary         all language-specific data, isolated from all logic
    TextNormalizer     script detection, accent folding, abbreviation expansion
    NegationDetector   scope windows and polarity
    ClinicalNoteLabeler  orchestrates the above

The split that matters most is Vocabulary vs the detectors. Adding a language
should mean adding data, never touching logic. If you find yourself editing
NegationDetector to support Polish, something is wrong with the boundary.
"""
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass, field
from typing import Iterable, Iterator, Sequence

__all__ = [
    "LABELS", "Certainty", "Mention", "LabelResult",
    "Vocabulary", "TextNormalizer", "NegationDetector", "ClinicalNoteLabeler",
]

# The 12 findings. Order is the submission column order; nothing else in this
# module hardcodes a label list.
LABELS: list[str] = [
    "ACL", "MCL", "medial_meniscus", "lateral_meniscus",
    "medial_OA", "lateral_OA", "patellofemoral_OA",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]


# =============================================================================
# 1. Certainty
# =============================================================================
class Certainty:
    """The ordinal scale a report can express about a finding.

    Six levels rather than a boolean, because reports hedge constantly and
    collapsing "definite tear" and "tear cannot be excluded" to the same 1
    throws away information the model can use.

    Categorical rather than a raw float because the mapping to numbers is a
    tunable you want in one place, calibrated against a gold set, not scattered
    through the matching code.
    """

    DEFINITE = "definite"
    PROBABLE = "probable"
    POSSIBLE = "possible"
    UNLIKELY = "unlikely"
    NEGATED = "negated"
    NOT_MENTIONED = "not_mentioned"
    UNSUPPORTED = "script_unsupported"     # see ClinicalNoteLabeler.extract

    # Strength ordering. Used to pick a winner when one report mentions the
    # same finding more than once.
    RANK = {
        DEFINITE: 5, PROBABLE: 4, POSSIBLE: 3,
        UNLIKELY: 2, NEGATED: 1, NOT_MENTIONED: 0, UNSUPPORTED: 0,
    }

    # Default categorical -> probability map. Override per project.
    DEFAULT_VALUES = {
        DEFINITE: 0.95, PROBABLE: 0.80, POSSIBLE: 0.50,
        UNLIKELY: 0.20, NEGATED: 0.02, NOT_MENTIONED: 0.02, UNSUPPORTED: 0.02,
    }

    @classmethod
    def stronger(cls, a: str, b: str) -> str:
        return a if cls.RANK[a] >= cls.RANK[b] else b


# =============================================================================
# 2/3. Value objects
# =============================================================================
@dataclass
class Mention:
    """One occurrence of one finding inside one text unit.

    Carries the unit it was found in so downstream code can show evidence
    without re-parsing, and so negation can be scoped without passing the whole
    document around.
    """

    label: str
    matched_text: str
    span: tuple[int, int]
    unit: str
    certainty: str = Certainty.DEFINITE
    negation_source: str = ""       # "", "pre", "post"

    def evidence(self, max_chars: int = 200) -> str:
        return self.unit.strip()[:max_chars]


@dataclass
class LabelResult:
    """Final verdict for one label on one report."""

    label: str
    certainty: str
    value: float
    evidence: str = ""
    needs_review: bool = False

    @property
    def is_positive(self) -> bool:
        return self.value > 0.5


# =============================================================================
# 4. Vocabulary — all language data, no logic
# =============================================================================
@dataclass
class Vocabulary:
    """Language-specific data for matching.

    TERM FORMAT: space-separated STEMS, not dictionary forms.
    "медиальн мениск" not "медиальный мениск". The compiler below turns each
    stem into `stem\\w*`, so every inflected form matches. This is essential for
    Russian and Greek, where an adjective-noun pair inflects on BOTH words and
    a literal substring search finds nothing.

    ACCURACY WARNING: the non-English entries below are a starting point, not
    validated terminology. Check them against your actual corpus before
    trusting them. A wrong stem produces no match, and no match looks exactly
    like a negative finding.
    """

    findings: dict[str, list[str]] = field(default_factory=dict)
    pre_negation: list[str] = field(default_factory=list)
    post_negation: list[str] = field(default_factory=list)
    hedges: dict[str, list[str]] = field(default_factory=dict)
    abbreviations: dict[str, str] = field(default_factory=dict)
    supported_scripts: set[str] = field(default_factory=lambda: {"latin", "greek", "cyrillic"})

    _compiled: dict[str, list[re.Pattern]] = field(default_factory=dict, repr=False)

    # -- compilation ------------------------------------------------------
    @staticmethod
    def compile_term(term: str) -> re.Pattern:
        r"""Turn "медиальн мениск" into `медиальн\w*[\s\-]*мениск\w*`.

        Each stem gets a trailing \w* so any case/number ending matches, and
        the separator tolerates a space or hyphen. Latin terms are unaffected:
        "medial meniscus" still matches itself, and now also "mediale
        meniscus" for free.
        """
        stems = [re.escape(s) for s in term.split()]
        return re.compile(r"\w*[\s\-]*".join(stems) + r"\w*", re.I | re.U)

    def compiled(self, label: str) -> list[re.Pattern]:
        """Lazily compile and cache patterns for one label."""
        if label not in self._compiled:
            self._compiled[label] = [self.compile_term(t) for t in self.findings.get(label, [])]
        return self._compiled[label]

    def add_language(self, findings: dict[str, list[str]],
                     pre: Sequence[str] = (), post: Sequence[str] = (),
                     hedges: dict[str, list[str]] | None = None) -> "Vocabulary":
        """Merge another language in. Returns self so calls chain.

        Scripts cannot collide -- a Cyrillic stem will never match Latin text --
        so everything lives in one merged pool and there is no routing by
        language at match time. That also means a mixed-language report works
        without any special handling.
        """
        for lab, terms in findings.items():
            self.findings.setdefault(lab, []).extend(terms)
        self.pre_negation.extend(pre)
        self.post_negation.extend(post)
        for bucket, cues in (hedges or {}).items():
            self.hedges.setdefault(bucket, []).extend(cues)
        self._compiled.clear()
        return self


def build_default_vocabulary() -> Vocabulary:
    """English/German/French/Spanish/Dutch + Greek + Russian."""
    v = Vocabulary(
        findings={
            "ACL": ["anterior cruciate"],
            "MCL": ["medial collateral"],
            "medial_meniscus": ["medial meniscus"],
            "lateral_meniscus": ["lateral meniscus"],
            "medial_OA": ["medial osteoarthritis", "medial compartment osteoarthritis",
                          "medial chondral loss"],
            "lateral_OA": ["lateral osteoarthritis", "lateral compartment osteoarthritis",
                           "lateral chondral loss"],
            "patellofemoral_OA": ["patellofemoral", "retropatellar", "chondromalacia patell"],
            "effusion": ["effusion", "joint fluid"],
            "synovitis": ["synovitis", "synovial thickening"],
            "bakers_cyst": ["baker cyst", "bakers cyst", "popliteal cyst"],
            "bone_contusion": ["bone marrow edema", "bone marrow oedema", "bone contusion",
                               "bone bruise"],
            "fracture": ["fracture", "avulsion"],
        },
        pre_negation=[r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b",
                      r"\babsence of\b", r"\bfree of\b", r"\bruled out\b", r"\bexcluded\b"],
        post_negation=[r"\bintact\b", r"\bnormal\b", r"\bunremarkable\b", r"\bpreserved\b",
                       r"\bwithin normal limits\b", r"\bwnl\b"],
        hedges={
            "probable": [r"likely", r"probable", r"consistent with", r"suggestive of"],
            "possible": [r"possible", r"cannot be (excluded|ruled out)", r"suspicion",
                         r"query", r"may represent", r"\bversus\b", r"\bvs\b"],
            "unlikely": [r"unlikely", r"doubtful"],
        },
        abbreviations={
            r"\bACL\b": "anterior cruciate ligament", r"\bVKB\b": "anterior cruciate ligament",
            r"\bLCA\b": "anterior cruciate ligament", r"\bMCL\b": "medial collateral ligament",
            r"\bMM\b": "medial meniscus", r"\bLM\b": "lateral meniscus",
            r"\bPF\b": "patellofemoral", r"\bOA\b": "osteoarthritis",
            r"\beff\b": "effusion", r"\bBME\b": "bone marrow edema",
            r"\bfx\b": "fracture", r"\bsyn\b": "synovitis",
        },
    )

    v.add_language(
        {   # German
            "ACL": ["vorder kreuzband", "kreuzband vorder"],
            "MCL": ["innenband", "mediale kollateralband"],
            "medial_meniscus": ["innenmeniskus"],
            "lateral_meniscus": ["aussenmeniskus", "außenmeniskus"],
            "medial_OA": ["mediale gonarthrose", "mediale arthrose"],
            "lateral_OA": ["laterale gonarthrose", "laterale arthrose"],
            "effusion": ["erguss", "gelenkerguss"],
            "bakers_cyst": ["bakerzyste"],
            "bone_contusion": ["knochenmarkodem", "knochenmarködem"],
            "fracture": ["fraktur"],
        },
        pre=[r"\bkein\b", r"\bkeine\b", r"\bohne\b"],
        post=[r"\bunauffallig\b", r"\bintakt\b", r"\bregelrecht\b"],
        hedges={"possible": [r"verdacht", r"moglich"], "probable": [r"wahrscheinlich"]},
    ).add_language(
        {   # French
            "ACL": ["ligament croise anterieur"],
            "medial_meniscus": ["menisque medial", "menisque interne"],
            "lateral_meniscus": ["menisque lateral", "menisque externe"],
            "effusion": ["epanchement"],
            "bakers_cyst": ["kyste de baker"],
            "bone_contusion": ["oedeme osseux"],
        },
        pre=[r"\bpas de\b", r"\bsans\b", r"\baucun\b"],
        post=[r"\bsans particularite\b", r"\bnormale?\b"],
        hedges={"possible": [r"possible", r"ne peut etre exclu", r"suspicion"],
                "probable": [r"compatible avec", r"en faveur de"]},
    ).add_language(
        {   # Spanish
            "ACL": ["ligamento cruzado anterior"],
            "medial_meniscus": ["menisco medial", "menisco interno"],
            "lateral_meniscus": ["menisco lateral", "menisco externo"],
            "effusion": ["derrame"],
            "bakers_cyst": ["quiste de baker"],
            "bone_contusion": ["edema oseo"],
            "fracture": ["fisura"],
        },
        pre=[r"\bno hay\b", r"\bsin\b", r"\bausencia\b"],
        post=[r"\bintacto\b", r"\bnormales?\b"],
        hedges={"possible": [r"posible", r"no se puede excluir", r"sospecha"],
                "probable": [r"compatible con", r"sugestivo de"]},
    ).add_language(
        {   # Dutch
            "ACL": ["voorste kruisband"],
            "medial_meniscus": ["mediale meniscus", "binnenmeniscus"],
            "lateral_meniscus": ["laterale meniscus", "buitenmeniscus"],
        },
        pre=[r"\bgeen\b", r"\bzonder\b"],
        post=[r"\bintact\b", r"\bnormaal\b"],
    ).add_language(
        {   # Russian (Cyrillic) -- stems
            "ACL": ["передн крестообразн", "пкс"],
            "MCL": ["внутренн боков", "медиальн боков"],
            "medial_meniscus": ["медиальн мениск", "внутренн мениск"],
            "lateral_meniscus": ["латеральн мениск", "наружн мениск"],
            "medial_OA": ["медиальн артроз", "внутренн артроз"],
            "lateral_OA": ["латеральн артроз", "наружн артроз"],
            "patellofemoral_OA": ["пателлофеморальн", "ретропателляр"],
            "effusion": ["выпот", "жидкост в полост"],
            "synovitis": ["синовит"],
            "bakers_cyst": ["киста бейкер", "подколенн киста"],
            "bone_contusion": ["отек костн мозг", "трабекулярн отек"],
            "fracture": ["перелом", "трещин"],
        },
        pre=[r"\bне\b", r"\bнет\b", r"\bбез\b"],
        # These are POST-posed in Russian: "перелом не выявлен" puts the
        # negation after the finding, unlike English "no fracture".
        post=[r"интактн", r"сохранн", r"не изменен", r"в норме", r"нормальн",
              r"не выявлен", r"не определя", r"не отмеча", r"не обнаружен", r"отсутств"],
        hedges={"possible": [r"возможн", r"подозрени", r"не исключ"],
                "probable": [r"вероятн", r"соответству"]},
    ).add_language(
        {   # Greek -- stems, written unaccented (normaliser strips tonos)
            "ACL": ["προσθι χιαστ", "χιαστου συνδεσμ"],
            "MCL": ["εσω πλαγι συνδεσμ"],
            "medial_meniscus": ["εσω μηνισκ", "εσωτερικ μηνισκ"],
            "lateral_meniscus": ["εξω μηνισκ", "εξωτερικ μηνισκ"],
            "medial_OA": ["εσω οστεοαρθρι"],
            "lateral_OA": ["εξω οστεοαρθρι"],
            "patellofemoral_OA": ["επιγονατιδομηριαι", "οπισθοεπιγονατιδ"],
            "effusion": ["αρθρικ συλλογ", "ενδαρθρικ υγρ"],
            "synovitis": ["υμενιτιδ"],
            "bakers_cyst": ["κυστ baker", "ιγνυακ κυστ"],
            "bone_contusion": ["οιδημα μυελ", "οστικ οιδημα"],
            "fracture": ["καταγμα", "ρωγμ"],
        },
        pre=[r"\bδεν\b", r"\bχωρις\b", r"απουσι", r"ουδεμι"],
        post=[r"ακεραι", r"φυσιολογικ", r"ανευ ευρηματ"],
        hedges={"possible": [r"πιθανον", r"δεν αποκλειετ"], "probable": [r"συμβατ"]},
    )
    return v


# =============================================================================
# 5. TextNormalizer
# =============================================================================
class TextNormalizer:
    """Script detection, accent folding, abbreviation expansion, unit splitting."""

    SCRIPT_RANGES = {
        "greek": (0x0370, 0x03FF), "cyrillic": (0x0400, 0x04FF),
        "arabic": (0x0600, 0x06FF), "hebrew": (0x0590, 0x05FF),
        "devanagari": (0x0900, 0x097F), "han": (0x4E00, 0x9FFF),
        "kana": (0x3040, 0x30FF), "hangul": (0xAC00, 0xD7AF),
    }

    # A unit boundary. Negation must not cross one.
    UNIT_SPLIT = re.compile(r"[\n\r]+|(?<=[.;:])\s+")

    def __init__(self, abbreviations: dict[str, str] | None = None):
        self.abbreviations = abbreviations or {}

    def detect_script(self, text: str) -> str:
        """Dominant script by letter census.

        Dominance, not presence: reports routinely mix Cyrillic prose with
        Latin units and drug names, so an any() test would misclassify them.
        """
        counts = {k: 0 for k in self.SCRIPT_RANGES}
        counts["latin"] = 0
        for ch in text or "":
            if not ch.isalpha():
                continue
            o = ord(ch)
            if o < 0x0250:
                counts["latin"] += 1
                continue
            for name, (lo, hi) in self.SCRIPT_RANGES.items():
                if lo <= o <= hi:
                    counts[name] += 1
                    break
        return max(counts, key=counts.get) if any(counts.values()) else "unknown"

    def normalize(self, text: str) -> str:
        """Fold accents, normalise Greek sigma, expand abbreviations, tidy space.

        NFKD decomposition plus combining-mark stripping folds Latin diacritics
        (é -> e) AND Greek tonos (ά -> α), which is exactly what we want since
        the Greek vocabulary is written unaccented. For Cyrillic it folds
        ё -> е and й -> и, harmless because the vocabulary goes through the
        same function.
        """
        if not text:
            return ""
        t = unicodedata.normalize("NFKD", text)
        t = "".join(c for c in t if not unicodedata.combining(c))
        t = t.replace("\u03c2", "\u03c3")        # Greek final sigma -> sigma
        for pat, rep in self.abbreviations.items():
            t = re.sub(pat, rep, t, flags=re.I)
        return re.sub(r"[ \t]+", " ", t)

    def split_units(self, text: str) -> list[str]:
        """Sentences or lines. The unit is the negation scope."""
        return [u.strip() for u in self.UNIT_SPLIT.split(text) if u.strip()]


# =============================================================================
# 6. NegationDetector
# =============================================================================
class NegationDetector:
    """Decides whether a mention is asserted, denied, or hedged.

    THE CENTRAL PROBLEM. "No evidence of ACL tear" contains "ACL tear". Since
    most mentions of most findings in radiology reports are negative, a matcher
    that cannot tell assertion from denial is worse than predicting the base
    rate.

    Two directions, because languages put the cue on different sides:
        pre-posed   "no fracture"            (English, German, French, Spanish)
        post-posed  "the ACL is intact"      (English)
                    "перелом не выявлен"     (Russian -- fracture not detected)

    A single backwards search gets post-posed negation exactly backwards,
    labelling intact structures as torn.
    """

    # Clause boundaries within a unit. "No fracture, but ACL torn" must not
    # negate the ACL.
    CLAUSE_BREAK = re.compile(
        r"[,;:]|\bbut\b|\bhowever\b|\baber\b|\bjedoch\b|\bmais\b|\bpero\b|\bmaar\b|\bно\b|\bαλλα\b",
        re.I | re.U,
    )

    def __init__(self, vocab: Vocabulary, window: int = 80):
        self.vocab = vocab
        self.window = window

    def _scopes(self, unit: str, span: tuple[int, int]) -> tuple[str, str]:
        """Left and right context, truncated at the nearest clause break."""
        left = unit[max(0, span[0] - self.window): span[0]].lower()
        breaks = list(self.CLAUSE_BREAK.finditer(left))
        if breaks:
            left = left[breaks[-1].end():]

        right = unit[span[1]: span[1] + self.window].lower()
        brk = self.CLAUSE_BREAK.search(right)
        if brk:
            right = right[: brk.start()]
        return left, right

    def detect(self, mention: Mention) -> str:
        """Returns '' (not negated), 'pre', or 'post'."""
        left, right = self._scopes(mention.unit, mention.span)
        if any(re.search(c, left, re.U) for c in self.vocab.pre_negation):
            return "pre"
        if any(re.search(c, right, re.U) for c in self.vocab.post_negation):
            return "post"
        return ""

    def certainty(self, mention: Mention) -> str:
        """Hedge level for a non-negated mention."""
        left, right = self._scopes(mention.unit, mention.span)
        ctx = left + " " + right
        for bucket, cues in self.vocab.hedges.items():
            if any(re.search(c, ctx, re.U) for c in cues):
                return bucket
        return Certainty.DEFINITE


# =============================================================================
# 7. ClinicalNoteLabeler
# =============================================================================
class ClinicalNoteLabeler:
    """Orchestrates normalisation, matching, negation and scoring.

        labeler = ClinicalNoteLabeler()
        results = labeler.extract(report)          # dict[label] -> LabelResult
        soft    = labeler.to_soft_labels(report)   # dict[label] -> float
        df_rows = list(labeler.batch(pairs))       # for a training CSV
    """

    def __init__(self, vocabulary: Vocabulary | None = None,
                 certainty_values: dict[str, float] | None = None,
                 omission_priors: dict[str, float] | None = None,
                 labels: Sequence[str] = LABELS):
        """
        certainty_values  categorical -> probability. Tune on a gold set.
        omission_priors   per-label p(present | never mentioned). See below.
        """
        self.vocab = vocabulary or build_default_vocabulary()
        self.values = dict(Certainty.DEFAULT_VALUES)
        if certainty_values:
            self.values.update(certainty_values)
        self.omission_priors = omission_priors or {}
        self.labels = list(labels)
        self.normalizer = TextNormalizer(self.vocab.abbreviations)
        self.negation = NegationDetector(self.vocab)

    # -- matching ---------------------------------------------------------
    def find_mentions(self, unit: str) -> list[Mention]:
        """All findings mentioned in one text unit.

        First matching term per label wins and we move on -- the terms within a
        label are synonyms, so a second hit adds nothing.
        """
        out = []
        for label in self.labels:
            for pat in self.vocab.compiled(label):
                m = pat.search(unit)
                if m:
                    out.append(Mention(label, m.group(0), m.span(), unit))
                    break
        return out

    # -- main entry point -------------------------------------------------
    def extract(self, text: str) -> dict[str, LabelResult]:
        script = self.normalizer.detect_script(text)
        normalized = self.normalizer.normalize(text)

        mentions: list[Mention] = []
        for unit in self.normalizer.split_units(normalized):
            for m in self.find_mentions(unit):
                src = self.negation.detect(m)
                m.negation_source = src
                m.certainty = Certainty.NEGATED if src else self.negation.certainty(m)
                mentions.append(m)

        # One report can mention a finding several times ("possible medial
        # meniscus tear" in findings, "medial meniscus tear" in impression).
        # The strongest assertion wins: a report that both hedges and asserts
        # has resolved its own uncertainty by the time it asserts.
        best: dict[str, Mention] = {}
        for m in mentions:
            cur = best.get(m.label)
            if cur is None or Certainty.RANK[m.certainty] > Certainty.RANK[cur.certainty]:
                best[m.label] = m

        # COVERAGE GUARD.
        # Zero mentions across a whole report means one of two things: a
        # genuinely unremarkable study, or a language this vocabulary does not
        # cover. Those produce identical output -- all negative -- and must not
        # be conflated, because the second silently poisons the training set
        # with an entire language's worth of false negatives. Flagging it turns
        # a silent failure into a visible one.
        unsupported = not mentions and script not in self.vocab.supported_scripts

        results = {}
        for label in self.labels:
            m = best.get(label)
            if m is None:
                cert = Certainty.UNSUPPORTED if unsupported else Certainty.NOT_MENTIONED
                val = self.omission_priors.get(label, self.values[Certainty.NOT_MENTIONED])
                results[label] = LabelResult(label, cert, val, "", unsupported)
            else:
                results[label] = LabelResult(
                    label, m.certainty, self.values[m.certainty], m.evidence(), False
                )
        return results

    def to_soft_labels(self, text: str) -> dict[str, float]:
        return {k: v.value for k, v in self.extract(text).items()}

    # -- batch helpers ----------------------------------------------------
    def batch(self, reports: Iterable[tuple[str, str]]) -> Iterator[dict]:
        """Yield one flat row per report. Feed straight to pd.DataFrame.

        A generator so a corpus of any size streams rather than materialising.
        """
        for study_id, text in reports:
            res = self.extract(text)
            yield {"study_id": study_id,
                   **{lab: r.value for lab, r in res.items()}}

    def review_queue(self, reports: Iterable[tuple[str, str]]) -> list[dict]:
        """Reports the extractor could not handle, for a human to look at.

        With a hand-built multilingual vocabulary this is the most useful
        diagnostic in the module: it tells you which languages or templates you
        are silently failing on, ranked so the worst come first.
        """
        rows = []
        for study_id, text in reports:
            res = self.extract(text)
            flagged = [r.label for r in res.values() if r.needs_review]
            if flagged:
                rows.append({
                    "study_id": study_id,
                    "script": self.normalizer.detect_script(text),
                    "n_flagged": len(flagged),
                    "preview": (text or "")[:80],
                })
        return sorted(rows, key=lambda r: -r["n_flagged"])

    def coverage_report(self, reports: Iterable[tuple[str, str]]) -> dict:
        """Per-script hit rate. Run this before trusting any output.

        A script with a near-zero mention rate is a vocabulary gap, not a
        population of healthy knees.
        """
        stats: dict[str, dict] = {}
        for _, text in reports:
            script = self.normalizer.detect_script(text)
            s = stats.setdefault(script, {"n": 0, "with_mentions": 0})
            s["n"] += 1
            res = self.extract(text)
            if any(r.certainty not in (Certainty.NOT_MENTIONED, Certainty.UNSUPPORTED)
                   for r in res.values()):
                s["with_mentions"] += 1
        for s in stats.values():
            s["hit_rate"] = round(s["with_mentions"] / max(s["n"], 1), 3)
        return stats


In [8]:
labeler = ClinicalNoteLabeler()
labeler.to_soft_labels("No Medial meniscus. No ACL tear found")

{'ACL': 0.02,
 'MCL': 0.02,
 'medial_meniscus': 0.02,
 'lateral_meniscus': 0.02,
 'medial_OA': 0.02,
 'lateral_OA': 0.02,
 'patellofemoral_OA': 0.02,
 'effusion': 0.02,
 'synovitis': 0.02,
 'bakers_cyst': 0.02,
 'bone_contusion': 0.02,
 'fracture': 0.02}

In [ ]:
# -*- coding: utf-8 -*-
"""Join the images to the reports and hand back one DataFrame.

    df = build_dataset(DATA_PATH, max_studies=50)

One row per series:

    study     StudyInstanceUID
    series    SeriesInstanceUID
    image     (n_slices, H, W) float array -- the DICOM pixels, in anatomical order
    <12 labels> soft labels parsed from that study's report, plus `labels`,
                the same twelve as one vector in LABELS order

...and the metadata you need to actually use those three: which plane and
weighting the series is, how big a voxel is, which knee, and the study id to
group on when you split.

MEMORY. The `image` column holds decoded pixels, so the frame is as big as the
data it loaded: one knee series is roughly 30 x 512 x 512 x 4 bytes = 31 MB,
and a study has several. The whole training set will not fit in RAM this way.
Three levers, in the order worth reaching for:

    max_studies=50          build on a subset while you develop
    image_dtype=np.uint16   quarter the size, lose the rescale correction
    load_images=False       identical frame, `paths` instead of `image`

The last one is what to use at full scale: keep the frame, read the pixels per
batch from `paths` with `read_volume()`.
"""
import os
from concurrent.futures import ThreadPoolExecutor
from glob import glob

import numpy as np
import pandas as pd
import pydicom


# =============================================================================
# Reading one series
# =============================================================================
def _tag(ds, *names, default=None):
    """First present, non-empty attribute. Vendors disagree on which tag holds what."""
    for n in names:
        v = getattr(ds, n, None)
        if v not in (None, ""):
            return v
    return default


def _f(v, default=None):
    """DICOM numbers arrive as DSfloat, IS, str, or a 1-element list."""
    if isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
        v = v[0] if len(v) else None
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


def _plane(ds):
    """'sagittal' | 'coronal' | 'axial' | 'oblique' | 'unknown'.

    Knee MRI is routinely prescribed oblique to the ligaments, so a normal that
    no axis clearly dominates is reported as oblique rather than rounded to the
    nearest plane and quietly mislabelled.
    """
    iop = _tag(ds, "ImageOrientationPatient")
    if iop is None or len(iop) != 6:
        return "unknown"
    iop = np.asarray(iop, float)
    n = np.abs(np.cross(iop[:3], iop[3:]))
    if not n.any():
        return "unknown"
    n = n / np.linalg.norm(n)
    axis = int(np.argmax(n))
    return ("sagittal", "coronal", "axial")[axis] if n[axis] >= 0.75 else "oblique"


def _weighting(description, ds):
    """T1 / T2 / PD / STIR / ... from the series name, falling back to timings.

    Knee protocols are named, not tagged: the weighting lives in
    SeriesDescription as free text a technologist typed. A guess, and treated
    as one -- use it to pick sequences, not as a feature.
    """
    t = (description or "").lower()
    for key, name in [("stir", "STIR"), ("t2*", "T2*"), ("pd", "PD"), ("proton", "PD"),
                      ("t1", "T1"), ("t2", "T2"), ("flair", "FLAIR"),
                      ("loc", "localizer"), ("scout", "localizer")]:
        if key in t:
            return name
    tr, te = _f(_tag(ds, "RepetitionTime")), _f(_tag(ds, "EchoTime"))
    if tr is not None and te is not None:
        if tr < 900 and te < 30:
            return "T1"
        if tr >= 2000:
            return "T2" if te >= 60 else "PD"
    return "unknown"


def order_slices(datasets, paths):
    """Paths sorted through-plane, the way the anatomy stacks.

    Position projected on the slice normal first, InstanceNumber second,
    filename last. Only the geometric sort is right for a series acquired
    interleaved; only the fallbacks work for a localizer with no position tags.
    Sorting by filename alone shuffles the stack on any scanner that does not
    zero-pad its exports.
    """
    iop = next((_tag(d, "ImageOrientationPatient") for d in datasets
                if _tag(d, "ImageOrientationPatient") is not None), None)
    if iop is not None and len(iop) == 6:
        iop = np.asarray(iop, float)
        normal = np.cross(iop[:3], iop[3:])
        pos = [_tag(d, "ImagePositionPatient") for d in datasets]
        if all(p is not None and len(p) == 3 for p in pos):
            proj = [float(np.dot(np.asarray(p, float), normal)) for p in pos]
            if len(set(proj)) > 1:
                return [p for _, p in sorted(zip(proj, paths))]

    nums = [_f(_tag(d, "InstanceNumber")) for d in datasets]
    if all(n is not None for n in nums) and len(set(nums)) > 1:
        return [p for _, p in sorted(zip(nums, paths))]
    return sorted(paths)


def read_volume(paths, dtype=np.float32, max_slices=None, indices=None):
    """(n_slices, H, W) from ordered paths.

    indices selects which slices to decode, and nothing else is read. Decoding
    a 200-slice series to keep 16 of it costs twelve times what it needs to,
    in both time and peak memory, and training only ever wants a sample.

    max_slices keeps the central N: the knee joint sits mid-stack, and the
    outermost slices are mostly air.
    """
    if indices is not None:
        paths = [paths[i] for i in indices]
    elif max_slices and len(paths) > max_slices:
        start = (len(paths) - max_slices) // 2
        paths = paths[start: start + max_slices]

    out = None
    for i, p in enumerate(paths):
        ds = pydicom.dcmread(p)
        arr = ds.pixel_array
        if out is None:
            # Preallocate: building a list and stacking it holds two full
            # copies of the volume at once, which is the peak that matters.
            out = np.empty((len(paths), *arr.shape), dtype=dtype)
        elif arr.shape != out.shape[1:]:
            raise ValueError(f"slices differ in shape: {arr.shape} vs {out.shape[1:]}")

        arr = arr.astype(np.float32, copy=False)
        slope = _f(_tag(ds, "RescaleSlope"), 1.0)
        intercept = _f(_tag(ds, "RescaleIntercept"), 0.0)
        if slope != 1.0 or intercept != 0.0:
            arr = arr * slope + intercept
        if _tag(ds, "PhotometricInterpretation") == "MONOCHROME1":
            arr = arr.max() - arr          # MONOCHROME1 stores bright-is-low
        out[i] = arr

    if out is None:
        raise ValueError("no slices to read")
    return out


def read_series(study, series, series_dir, load_images=True,
                image_dtype=np.float32, max_slices=None):
    """One series -> one row. Never raises: a broken series comes back flagged.

    A scan that dies on file 40,000 of 50,000 has cost you the other 49,999, so
    every failure lands in `error` and the run continues.
    """
    row = {"study": study, "series": series, "image": None, "paths": (),
           "n_slices": 0, "n_unreadable": 0, "error": ""}

    paths = sorted(glob(os.path.join(series_dir, "*.dcm")))
    if not paths:
        row["error"] = "no .dcm files"
        return row

    datasets, kept = [], []
    for p in paths:
        try:
            datasets.append(pydicom.dcmread(p, stop_before_pixels=True))
            kept.append(p)
        except Exception:
            row["n_unreadable"] += 1        # one bad file, not a bad series
    if not datasets:
        row["error"] = f"all {len(paths)} files unreadable"
        return row

    by_path = dict(zip(kept, datasets))
    ordered = order_slices(datasets, kept)
    head = by_path[ordered[0]]

    # Rows/Columns are in the header, so a series that cannot be stacked is
    # caught whether or not pixels get read -- otherwise load_images=False
    # hands back a row that blows up later, in the middle of training.
    shapes = {(int(_f(_tag(by_path[p], "Rows"), 0)),
               int(_f(_tag(by_path[p], "Columns"), 0))) for p in ordered}
    if len(shapes) > 1:
        row["error"] = f"slices differ in shape: {sorted(shapes)}"

    spacing = _tag(head, "PixelSpacing", default=[None, None])
    desc = str(_tag(head, "SeriesDescription", "ProtocolName", default=""))
    age = str(_tag(head, "PatientAge", default=""))

    row.update({
        "paths": tuple(ordered),
        "n_slices": len(ordered),
        "n_unreadable": row["n_unreadable"],
        "rows": int(_f(_tag(head, "Rows"), 0)) or None,
        "cols": int(_f(_tag(head, "Columns"), 0)) or None,
        # plane and weighting are how you choose which series to feed a model:
        # menisci are read on sagittal fat-sat, collaterals on coronal.
        "plane": _plane(head),
        "weighting": _weighting(desc, head),
        "fat_sat": any(k in desc.lower() for k in ("fs", "fat sat", "stir", "spair", "spir")),
        "series_description": desc,
        # geometry, so volumes can be resampled to a common physical size
        "pixel_spacing": _f(spacing[0]),
        "slice_thickness": _f(_tag(head, "SliceThickness")),
        "spacing_between_slices": _f(_tag(head, "SpacingBetweenSlices")),
        "laterality": str(_tag(head, "ImageLaterality", "Laterality", default="")),
        "patient_age": _f(age[:-1]) if age[-1:].upper() == "Y" else _f(age),
        "patient_sex": str(_tag(head, "PatientSex", default="")),
        "manufacturer": str(_tag(head, "Manufacturer", default="")),
    })

    if load_images and not row["error"]:
        try:
            row["image"] = read_volume(ordered, image_dtype, max_slices)
        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"
    return row


def drop_images(df):
    """Free the `image` column, keeping everything else.

    For when the pixels were loaded to look at and are now just sitting in
    memory. `paths` stays, so the training code reads them per batch and
    nothing downstream notices.
    """
    out = df.copy()
    out["image"] = None
    return out


# =============================================================================
# Reading the reports
# =============================================================================
REPORT_COLUMNS = ("report", "report_text", "radiology_report", "clinical_notes",
                  "doctor_notes", "notes", "text", "findings", "impression",
                  "conclusion")


def read_reports(data_path, split="train", report_col=None, labeler=None):
    """train.csv -> one row per study, with the 12 soft labels attached.

    The report column is auto-detected as a first-run convenience; pass
    report_col once you know the name. `needs_review` marks reports in a script
    the vocabulary cannot read -- those come out all-negative for the wrong
    reason, and training on them as negatives poisons the label set.
    """
    df = pd.read_csv(os.path.join(data_path, f"{split}.csv"))
    labeler = labeler or ClinicalNoteLabeler()

    if report_col:
        cols = [report_col] if isinstance(report_col, str) else list(report_col)
    else:
        lower = {c.lower(): c for c in df.columns}
        cols = [lower[n] for n in REPORT_COLUMNS if n in lower]
    if not cols:
        raise KeyError(f"No report column in {list(df.columns)}. Pass report_col='<name>'.")

    text = (df[cols].fillna("").astype(str)
            .apply(lambda r: "\n".join(x for x in r if x.strip()), axis=1))

    rows = []
    for t in text:
        res = labeler.extract(t)
        rows.append({
            **{lab: res[lab].value for lab in LABELS},
            "labels": np.array([res[lab].value for lab in LABELS], dtype=np.float32),
            "report": t,
            "report_script": labeler.normalizer.detect_script(t),
            "needs_review": any(r.needs_review for r in res.values()),
        })

    out = pd.concat([df[["StudyInstanceUID"]].reset_index(drop=True),
                     pd.DataFrame(rows)], axis=1)
    return out.rename(columns={"StudyInstanceUID": "study"})


# =============================================================================
# The join
# =============================================================================
def estimate_pixel_bytes(series_rows, image_dtype=np.float32, max_slices=None):
    """How much RAM the `image` column would take, from the headers alone.

    Cheap to compute and worth knowing before the fact: the alternative is
    finding out when the kernel is killed twenty minutes in.
    """
    itemsize = np.dtype(image_dtype).itemsize
    total = 0
    for r in series_rows:
        n = min(r.get("n_slices", 0), max_slices) if max_slices else r.get("n_slices", 0)
        total += n * (r.get("rows") or 0) * (r.get("cols") or 0) * itemsize
    return total


def build_dataset(data_path, split="train", *, report_col=None, labeler=None,
                  load_images=True, image_dtype=np.float32, max_slices=None,
                  max_studies=None, workers=8, drop_failed=True,
                  memory_budget_gb=4.0, verbose=True):
    """One DataFrame: study, series, image, labels, and the metadata to use them.

    max_studies      build on a subset -- see the memory note at the top
    max_slices       keep the central N slices of each series
    load_images      False fills `paths` instead of `image`, at full scale
    memory_budget_gb refuse to hold more than this in the `image` column; the
                     headers are read first, so the size is known before any
                     pixel is decoded, and the build falls back to paths
                     instead of being killed part way through
    drop_failed      leave out series that could not be read (they are listed
                     either way when verbose)
    """
    index = pd.read_csv(os.path.join(data_path, f"{split}_series.csv"))
    reports = read_reports(data_path, split, report_col, labeler)

    studies = list(dict.fromkeys(index["StudyInstanceUID"]))
    if max_studies:
        studies = studies[:max_studies]
        index = index[index["StudyInstanceUID"].isin(set(studies))]

    jobs = [(s, ser, os.path.join(data_path, f"{split}_series", s, ser))
            for s, ser in zip(index["StudyInstanceUID"], index["SeriesInstanceUID"])]
    if verbose:
        print(f"reading {len(jobs)} series from {len(studies)} studies"
              f"{' with pixels' if load_images else ' (metadata only)'} ...")

    # Phase 1: headers only. Cheap, and it tells us what the pixels would cost.
    with ThreadPoolExecutor(max_workers=workers) as pool:
        rows = list(pool.map(
            lambda j: read_series(*j, load_images=False,
                                  image_dtype=image_dtype, max_slices=max_slices),
            jobs))

    # Phase 2: pixels, if they fit.
    if load_images:
        want = estimate_pixel_bytes(rows, image_dtype, max_slices)
        budget = memory_budget_gb * 1e9
        if want > budget:
            load_images = False
            print(f"the image column would need {want/1e9:,.1f} GB, over the "
                  f"{memory_budget_gb:g} GB budget -- keeping `paths` instead.\n"
                  f"  the training code reads pixels per batch from `paths`, so "
                  f"this costs nothing except `df['image']` being empty.\n"
                  f"  to load them anyway: raise memory_budget_gb, or cut the "
                  f"data with max_studies= / max_slices=.")
        else:
            if verbose:
                print(f"  loading pixels (~{want/1e9:.2f} GB) ...")
            for r in rows:
                if r["n_slices"] and not r["error"]:
                    try:
                        r["image"] = read_volume(r["paths"], image_dtype, max_slices)
                    except Exception as exc:
                        r["error"] = f"{type(exc).__name__}: {exc}"

    series = pd.DataFrame(rows)
    failed = series[series["error"] != ""]
    if verbose and len(failed):
        print(f"  {len(failed)} series had problems:")
        for _, r in failed.head(5).iterrows():
            print(f"    {r['series']}: {r['error']}")
    if drop_failed:
        series = series[(series["error"] == "") & (series["n_slices"] > 0)]

    # Left join from the series side: a label with no pixels behind it is not a
    # training row. Labels are per study and broadcast to each of its series.
    df = series.merge(reports, on="study", how="left")
    df["n_series_in_study"] = df.groupby("study")["series"].transform("count")
    # Group by this when you split. Every series of a study shares one report,
    # so a row-level split puts the same label on both sides and flatters CV.
    df["group"] = df["study"]

    front = ["study", "series", "image", "labels", *LABELS,
             "n_slices", "rows", "cols", "plane", "weighting", "fat_sat",
             "laterality", "pixel_spacing", "slice_thickness"]
    df = df[[c for c in front if c in df] + [c for c in df.columns if c not in front]]

    if verbose:
        mb = df.memory_usage(deep=True).sum() / 1e6
        if load_images:
            mb += sum(im.nbytes for im in df["image"] if im is not None) / 1e6
        print(f"{len(df)} series x {len(df.columns)} columns, ~{mb:,.0f} MB in memory")
    return df.reset_index(drop=True)

In [ ]:
# ============================================================================
# Build it
# ============================================================================
df = build_dataset(DATA_PATH, max_studies=50)     # drop max_studies for all of it

df[["study", "series", "image", "labels"]].head()


In [ ]:
# One row, end to end.
row = df.iloc[0]
print(row["study"], row["series"])
print("image ", row["image"].shape, row["image"].dtype)      # (n_slices, H, W)
print("labels", dict(zip(LABELS, row["labels"].round(2))))
print("series", row["plane"], row["weighting"], "fat_sat" if row["fat_sat"] else "")

# Check the labels before training on them: a report in a script the
# vocabulary cannot read comes out all-negative, which is indistinguishable
# from a normal knee unless you look.
print("\nflagged reports:", df["needs_review"].sum())
display(df[df["needs_review"]][["study", "report_script", "report"]].drop_duplicates("study"))

# Sagittal fat-sat is what the menisci are read on.
sag = df[(df["plane"] == "sagittal") & df["fat_sat"]]
print(len(sag), "sagittal fat-sat series")

# Split on `group` (the study), never on the row: every series of a study
# shares one report, so a row-level split puts the same label on both sides.
#   from sklearn.model_selection import GroupShuffleSplit
#   tr, va = next(GroupShuffleSplit(test_size=0.2, random_state=0)
#                 .split(df, groups=df["group"]))


In [ ]:
# -*- coding: utf-8 -*-
"""Test / train / validation splits, grouped by study.

    df = make_splits(df, n_folds=5, test_frac=0.15)
    df[["study", "series", "split", "fold"]]

THE ONE RULE. A study's series all share one report, so they all carry the
same twelve labels. Split rows and the same label lands on both sides of the
boundary: the model sees a sagittal slice of a knee in training and a coronal
slice of the *same* knee in validation, and your CV score measures memory, not
generalisation. So studies are split, never rows, and the row assignment is
joined back afterwards -- which makes leakage structurally impossible rather
than something to remember to check.

STRATIFICATION. Eight of the twelve findings are rare, and with a few hundred
studies a plain random split will hand some fold zero positives for a label,
which makes that label's AUC undefined and the fold means incomparable. Each
study is therefore keyed by the *rarest* finding it has, and that key is
balanced across the splits. It is a heuristic -- true multi-label
stratification needs iterative stratification (`iterstrat`) -- but it is the
one that protects the labels most at risk, which is what actually breaks.
"""
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold


def study_strata(df, labels=None, threshold=0.5, study_col="study"):
    """One stratification key per study: the rarest finding it is positive for.

    Returns a frame of (study, stratum). "none" for a study with no positive
    finding, which is itself a class worth balancing -- normal knees are a
    large fraction of the set.
    """
    labels = list(labels if labels is not None else LABELS)
    studies = df.drop_duplicates(study_col).set_index(study_col)[labels]
    positive = studies > threshold

    # Rarest first, so the label most likely to vanish from a fold is the one
    # that decides the key.
    prevalence = positive.mean().sort_values()
    order = list(prevalence.index)

    def key(row):
        for lab in order:                 # rarest -> commonest
            if row[lab]:
                return lab
        return "none"

    return pd.DataFrame({
        study_col: studies.index,
        "stratum": positive.apply(key, axis=1).values,
    })


def _collapse_rare(strata, min_count):
    """Fold strata too small to split into one 'rare' bucket.

    StratifiedKFold warns (and stratifies badly) when a class has fewer members
    than folds. Merging the small ones keeps the split working and still
    separates those studies from the rest -- but the merged bucket can itself
    still be too small, so it is then folded into the largest class rather than
    left to trip the same warning.
    """
    strata = strata.copy()
    counts = strata["stratum"].value_counts()
    small = set(counts[counts < min_count].index)
    if not small:
        return strata

    strata["stratum"] = strata["stratum"].where(~strata["stratum"].isin(small), "rare")
    counts = strata["stratum"].value_counts()
    if counts.get("rare", 0) < min_count and len(counts) > 1:
        biggest = counts.drop(index="rare").idxmax()
        strata["stratum"] = strata["stratum"].replace("rare", biggest)
    return strata


def make_splits(df, n_folds=5, test_frac=0.15, seed=42, labels=None,
                threshold=0.5, drop_needs_review=True, study_col="study",
                verbose=True):
    """Add `split` ('train'/'test') and `fold` (0..n_folds-1, -1 for test).

    Validation is fold k of the training rows: it is not a third fixed set,
    because with a few hundred studies a fixed validation set is small enough
    that model selection on it is mostly noise. Cross-validation reuses every
    study for validation exactly once.

    drop_needs_review removes studies whose report the labeler could not read.
    Their labels are all-negative for the wrong reason, so training on them
    teaches the model that an abnormal knee is normal. Set False to keep them.
    """
    labels = list(labels if labels is not None else LABELS)
    out = df.copy()

    if drop_needs_review and "needs_review" in out.columns:
        flagged = out["needs_review"].fillna(False).astype(bool)
        if verbose and flagged.any():
            print(f"dropping {flagged.sum()} rows "
                  f"({out.loc[flagged, study_col].nunique()} studies) flagged needs_review")
        out = out[~flagged]

    strata = study_strata(out, labels, threshold, study_col)

    # --- hold out the test studies ---------------------------------------
    n_test_splits = max(2, int(round(1 / test_frac)))
    s = _collapse_rare(strata, n_test_splits)
    splitter = StratifiedKFold(n_splits=n_test_splits, shuffle=True, random_state=seed)
    rest_idx, test_idx = next(splitter.split(s, s["stratum"]))
    test_studies = set(s.iloc[test_idx][study_col])
    rest = s.iloc[rest_idx].reset_index(drop=True)

    # --- fold the remainder ----------------------------------------------
    rest = _collapse_rare(rest, n_folds)
    folder = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_of = {}
    for k, (_, val_idx) in enumerate(folder.split(rest, rest["stratum"])):
        for study in rest.iloc[val_idx][study_col]:
            fold_of[study] = k

    out["split"] = np.where(out[study_col].isin(test_studies), "test", "train")
    out["fold"] = out[study_col].map(fold_of).fillna(-1).astype(int)

    # Structural guarantee, asserted rather than assumed.
    overlap = set(out.loc[out["split"] == "test", study_col]) & \
              set(out.loc[out["split"] == "train", study_col])
    assert not overlap, f"study in both splits: {sorted(overlap)[:3]}"

    if verbose:
        print(f"test  {(out['split'] == 'test').sum():5d} series "
              f"({out.loc[out['split'] == 'test', study_col].nunique()} studies)")
        for k in range(n_folds):
            rows = out[(out["split"] == "train") & (out["fold"] == k)]
            print(f"fold {k} {len(rows):5d} series ({rows[study_col].nunique()} studies) "
                  f"-- validation when training fold {k}")
    return out


def fold_frames(df, fold):
    """(train, val) for one fold. Test stays out of both."""
    train_rows = df[(df["split"] == "train") & (df["fold"] != fold)]
    val_rows = df[(df["split"] == "train") & (df["fold"] == fold)]
    return train_rows.reset_index(drop=True), val_rows.reset_index(drop=True)

In [ ]:
# -*- coding: utf-8 -*-
"""Train DINOv2 on the volumes. Images and labels only -- no metadata features.

    df = make_splits(build_dataset(DATA_PATH, max_studies=200))
    oof, summary = cross_validate(df, TrainConfig(epochs=8))

HOW A 2D MODEL EATS A 3D SERIES.
DINOv2 is a 2D ViT: it has no notion of a stack. A series is therefore fed as
N slices, each embedded independently, and the N embeddings are pooled into one
series vector before the head. The pooling is learned attention rather than a
mean, because a finding lives on a handful of slices -- a meniscal tear is two
or three of forty -- and a mean over forty slices divides that signal by forty.
Attention lets the model put its weight where the tear is, and the weights are
readable afterwards, which is the cheapest interpretability you will get.

WHAT IS NOT AUGMENTED, AND WHY.
No horizontal flips. On a coronal knee a flip swaps the medial and lateral
compartments, so `medial_OA` and `lateral_OA` become each other's labels; on a
sagittal knee it swaps anterior for posterior, which does the same to the
cruciates. The usual reflex augmentation silently corrupts four of the twelve
labels here. Intensity and mild crop jitter are safe and are used instead.

THE BACKBONE IS FROZEN BY DEFAULT.
A few hundred studies against 21M parameters fine-tunes straight into
memorisation. Frozen DINOv2 features plus a trained head is the honest
baseline, and it is what you should beat before unfreezing anything. When the
backbone is frozen the slice embeddings never change, so precomputing them once
turns each epoch from minutes into seconds -- worth doing as soon as you are
past the first run.
"""
import gc
import math
import os
import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset


# =============================================================================
# Config
# =============================================================================
@dataclass
class TrainConfig:
    # data
    n_slices: int = 16              # slices sampled per series
    image_size: int = 224           # must be a multiple of 14 for a /14 ViT
    batch_size: int = 4             # volumes, so n_slices x this images per step
    num_workers: int = 2

    # model
    model_name: str = "vit_small_patch14_dinov2.lvd142m"
    pretrained: bool = True
    in_chans: int = 1               # gray in, no RGB replication
    freeze_backbone: bool = True
    unfreeze_last_n: int = 0        # last N transformer blocks, if not frozen
    dropout: float = 0.1

    # optimisation
    epochs: int = 8
    head_lr: float = 1e-3
    backbone_lr: float = 1e-5       # only used when something is unfrozen
    weight_decay: float = 1e-4
    warmup_frac: float = 0.1
    patience: int = 3               # early stop after N epochs with no gain
    grad_clip: float = 1.0

    seed: int = 42
    amp: bool = True                # ignored on CPU
    out_dir: str = "checkpoints"
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")


def free_memory():
    """Return what was just dropped, instead of waiting for the collector.

    Worth calling between folds: Python frees the tensors when it gets round
    to it, and the next fold allocates before that happens.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gpu_report(prefix=""):
    """Current GPU allocation, when there is a GPU."""
    if not torch.cuda.is_available():
        return
    used = torch.cuda.memory_allocated() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{prefix}GPU {used:.2f} GB in use, {peak:.2f} GB peak, {total:.0f} GB total")


def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =============================================================================
# Dataset -- image and label, nothing else
# =============================================================================
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
# One channel, not three. Replicating grayscale to RGB triples every tensor
# from the worker through pinned memory and across the bus, to carry the same
# number three times. The backbone's patch embedding is folded to one input
# channel instead (see load_backbone), which is what timm does for in_chans=1
# and is equivalent for a gray image. These are the ImageNet statistics
# averaged over the three channels.
GRAY_MEAN, GRAY_STD = 0.449, 0.226


def pick_slices(n_available, n_wanted, jitter=False, rng=None):
    """Evenly spaced slice indices, optionally jittered within each interval.

    Even spacing rather than the middle N: a knee series covers the whole joint
    and a finding can sit anywhere in it. Jitter makes the sampled set differ
    between epochs, which is augmentation the anatomy cannot object to.
    """
    if n_available <= n_wanted:
        idx = list(range(n_available))
        return idx + [n_available - 1] * (n_wanted - n_available)   # pad by repeat

    edges = np.linspace(0, n_available, n_wanted + 1)
    if jitter and rng is not None:
        return [int(min(n_available - 1, rng.uniform(edges[i], edges[i + 1])))
                for i in range(n_wanted)]
    return [int((edges[i] + edges[i + 1]) / 2) for i in range(n_wanted)]


def normalize_volume(vol):
    """Per-volume robust scaling to [0, 1].

    Per volume because MR intensities have no physical units -- they shift with
    coil, scanner and sequence, so a global constant is meaningless. Robust
    (1st/99th percentile) because a single bright artefact otherwise compresses
    the entire knee into the bottom of the range.
    """
    lo, hi = np.percentile(vol, (1.0, 99.0))
    if hi <= lo:
        lo, hi = float(vol.min()), float(vol.max())
    if hi <= lo:
        return np.zeros_like(vol, dtype=np.float32)
    return np.clip((vol - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)


class KneeVolumeDataset(Dataset):
    """Yields (slices, labels) and nothing else.

    slices  float tensor (n_slices, 3, size, size), ImageNet-normalised
    labels  float tensor (12,) -- soft targets in [0, 1]

    Grayscale is replicated across three channels: DINOv2 was trained on RGB
    and its patch embedding expects three, and replication keeps the
    pretrained filters in the regime they were fitted on.
    """

    def __init__(self, df, labels, n_slices=16, image_size=224, train=False, seed=0):
        self.rows = df.reset_index(drop=True)
        self.labels = list(labels)
        self.n_slices = n_slices
        self.image_size = image_size
        self.train = train
        self.seed = seed

    def __len__(self):
        return len(self.rows)

    def _slices(self, row, idx):
        """Only the sampled slices, decoded only if they are not already here.

        The order matters: pick the indices, then read. Reading the whole
        series and subscripting it decodes every slice to throw most away --
        on a 200-slice series that is twelve times the IO and twelve times the
        peak memory, per sample, every epoch.
        """
        img = row["image"]
        if img is not None and not (isinstance(img, float) and np.isnan(img)):
            return np.asarray(img[idx], dtype=np.float32)
        return read_volume(row["paths"], indices=idx)

    def __getitem__(self, i):
        row = self.rows.iloc[i]
        rng = np.random.default_rng(self.seed + i if not self.train else None)

        img = row["image"]
        have_pixels = img is not None and not (isinstance(img, float) and np.isnan(img))
        n_available = len(img) if have_pixels else int(row["n_slices"] or len(row["paths"]))
        idx = pick_slices(n_available, self.n_slices, jitter=self.train, rng=rng)

        # Scaled on the sampled slices rather than the whole series: the whole
        # series is what we are avoiding reading. The percentiles are close
        # enough on a dozen slices spanning the joint.
        vol = normalize_volume(self._slices(row, idx))
        x = torch.from_numpy(vol)                            # (S, H, W)

        if self.train:
            # Intensity jitter: scanners vary more than this between sites.
            x = torch.clamp(x * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05), 0, 1)

        x = x.unsqueeze(1)                                   # (S, 1, H, W)
        x = F.interpolate(x, size=(self.image_size, self.image_size),
                          mode="bilinear", align_corners=False)
        x = (x - GRAY_MEAN) / GRAY_STD

        y = torch.tensor([float(row[l]) for l in self.labels], dtype=torch.float32)
        return x, y


# =============================================================================
# Model
# =============================================================================
def load_backbone(model_name, image_size, pretrained=True, in_chans=1):
    """DINOv2 via timm, then torch.hub, then a local checkpoint.

    Three routes because the weights come from the internet and Kaggle
    notebooks often have none: with the internet off, attach the DINOv2 weights
    as a dataset and point $DINOV2_CHECKPOINT at the .pth.
    """
    import timm

    try:
        # in_chans=1 makes timm sum the pretrained patch-embedding weights
        # across RGB, which is exactly right for a gray image and removes the
        # three-fold copy from every tensor in the pipeline.
        model = timm.create_model(model_name, pretrained=pretrained, num_classes=0,
                                  img_size=image_size, in_chans=in_chans)
        return model, model.num_features
    except Exception as exc:
        print(f"timm could not fetch {model_name}: {type(exc).__name__}: {exc}")

    local = os.environ.get("DINOV2_CHECKPOINT")
    if local and os.path.exists(local):
        model = timm.create_model(model_name, pretrained=False, num_classes=0,
                                  img_size=image_size, in_chans=in_chans)
        state = torch.load(local, map_location="cpu")
        state = _fold_patch_embed(state.get("model", state), in_chans)
        missing, unexpected = model.load_state_dict(
            state.get("model", state), strict=False)
        print(f"loaded {local} (missing {len(missing)}, unexpected {len(unexpected)})")
        return model, model.num_features

    try:
        hub = {"vit_small_patch14_dinov2.lvd142m": "dinov2_vits14",
               "vit_base_patch14_dinov2.lvd142m": "dinov2_vitb14"}.get(model_name, "dinov2_vits14")
        model = torch.hub.load("facebookresearch/dinov2", hub)
        if in_chans == 1:
            model.load_state_dict(_fold_patch_embed(model.state_dict(), 1), strict=False)
            model.patch_embed.proj = _folded_conv(model.patch_embed.proj)
        return model, model.embed_dim
    except Exception as exc:
        raise RuntimeError(
            f"Could not load DINOv2 weights ({type(exc).__name__}: {exc}). "
            "With the internet off, attach the weights as a Kaggle dataset and "
            "set $DINOV2_CHECKPOINT, or pass pretrained=False to train from "
            "scratch (which will not work well on this much data)."
        ) from exc


def _fold_patch_embed(state, in_chans):
    """Sum a 3-channel patch-embedding kernel down to `in_chans`.

    A gray image repeated three times and convolved with three kernels gives
    exactly the same result as the image convolved with their sum, so this
    loses nothing and saves carrying the copies.
    """
    if in_chans != 1:
        return state
    out = dict(state)
    for k, v in state.items():
        if k.endswith("patch_embed.proj.weight") and v.ndim == 4 and v.shape[1] == 3:
            out[k] = v.sum(dim=1, keepdim=True)
    return out


def _folded_conv(conv):
    """The same conv with one input channel and summed weights."""
    folded = nn.Conv2d(1, conv.out_channels, conv.kernel_size, conv.stride,
                       conv.padding, bias=conv.bias is not None)
    with torch.no_grad():
        folded.weight.copy_(conv.weight.sum(dim=1, keepdim=True))
        if conv.bias is not None:
            folded.bias.copy_(conv.bias)
    return folded


class AttentionPool(nn.Module):
    """Pool slice embeddings into one series embedding, with learned weights.

    Returns the weights as well: `w[i]` is how much slice i mattered, which
    plots straight onto the stack and tells you whether the model is looking
    at the joint or at the edge of the field of view.
    """

    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 128),
                                   nn.Tanh(), nn.Linear(128, 1))

    def forward(self, x):                        # x: (B, S, D)
        w = torch.softmax(self.score(x), dim=1)  # (B, S, 1)
        return (x * w).sum(dim=1), w.squeeze(-1)


class DinoV2MultiLabel(nn.Module):
    """DINOv2 per slice -> attention pool over slices -> 12 logits."""

    def __init__(self, n_labels, cfg: TrainConfig):
        super().__init__()
        self.backbone, dim = load_backbone(cfg.model_name, cfg.image_size,
                                           cfg.pretrained, cfg.in_chans)

        if cfg.freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
            if cfg.unfreeze_last_n:
                blocks = getattr(self.backbone, "blocks", [])
                for blk in list(blocks)[-cfg.unfreeze_last_n:]:
                    for p in blk.parameters():
                        p.requires_grad = True

        self.pool = AttentionPool(dim)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(cfg.dropout),
                                  nn.Linear(dim, n_labels))
        self.frozen = cfg.freeze_backbone and not cfg.unfreeze_last_n

    def forward(self, x, return_weights=False):  # x: (B, S, 3, H, W)
        b, s = x.shape[:2]
        flat = x.flatten(0, 1)
        # No grad through a fully frozen backbone: it halves memory and lets
        # the batch be twice the size.
        with torch.set_grad_enabled(self.training and not self.frozen):
            feats = self.backbone(flat)
        feats = feats.reshape(b, s, -1).float()
        pooled, weights = self.pool(feats)
        logits = self.head(pooled)
        return (logits, weights) if return_weights else logits


# =============================================================================
# Metrics
# =============================================================================
def multilabel_metrics(y_true, y_prob, labels, threshold=0.5):
    """Macro AUC and macro average precision over the labels that have both classes.

    Labels with no positive (or no negative) in a fold have no defined AUC.
    They are skipped and counted rather than scored as 0.5, which would drag a
    fold mean around for a reason that has nothing to do with the model.
    """
    y_bin = (y_true > threshold).astype(int)
    aucs, aps, skipped = {}, {}, []
    for i, lab in enumerate(labels):
        col = y_bin[:, i]
        if col.min() == col.max():
            skipped.append(lab)
            continue
        aucs[lab] = roc_auc_score(col, y_prob[:, i])
        aps[lab] = average_precision_score(col, y_prob[:, i])
    return {
        "auc": float(np.mean(list(aucs.values()))) if aucs else float("nan"),
        "ap": float(np.mean(list(aps.values()))) if aps else float("nan"),
        "per_label_auc": aucs,
        "skipped": skipped,
    }


# =============================================================================
# Train / evaluate
# =============================================================================
def run_epoch(model, loader, loss_fn, cfg, optimizer=None, scheduler=None, scaler=None):
    """One pass. Optimiser given -> train; omitted -> evaluate."""
    train = optimizer is not None
    model.train(train)
    device = cfg.device
    use_amp = cfg.amp and device == "cuda"

    total, n, preds, targets = 0.0, 0, [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = loss_fn(logits, y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()

        total += float(loss) * len(y)
        n += len(y)
        preds.append(torch.sigmoid(logits.detach().float()).cpu().numpy())
        targets.append(y.detach().cpu().numpy())

    return total / max(n, 1), np.concatenate(preds), np.concatenate(targets)


def make_loader(df, labels, cfg, train):
    ds = KneeVolumeDataset(df, labels, cfg.n_slices, cfg.image_size, train, cfg.seed)
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=train,
                      num_workers=cfg.num_workers, pin_memory=(cfg.device == "cuda"),
                      drop_last=False)


def train_fold(df, fold, labels, cfg: TrainConfig, verbose=True):
    """Train one fold. Returns (checkpoint_path, history, best_metrics, oof).

    The trained weights come back as a path rather than a module: they are
    written every time validation improves anyway, and keeping five folds'
    worth of live model is how a 16 GB GPU runs out on the fourth fold.
    """
    seed_everything(cfg.seed + fold)
    os.makedirs(cfg.out_dir, exist_ok=True)
    ckpt_path = os.path.join(cfg.out_dir, f"fold{fold}.pt")

    train_df, val_df = fold_frames(df, fold)
    train_loader = make_loader(train_df, labels, cfg, train=True)
    val_loader = make_loader(val_df, labels, cfg, train=False)

    model = DinoV2MultiLabel(len(labels), cfg).to(cfg.device)

    # Two learning rates: a head from scratch wants a large one, a pretrained
    # backbone wants a small one or it forgets what it knows in one epoch.
    head_params = [p for n_, p in model.named_parameters()
                   if p.requires_grad and not n_.startswith("backbone")]
    back_params = [p for n_, p in model.named_parameters()
                   if p.requires_grad and n_.startswith("backbone")]
    groups = [{"params": head_params, "lr": cfg.head_lr}]
    if back_params:
        groups.append({"params": back_params, "lr": cfg.backbone_lr})
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)

    steps = max(1, len(train_loader)) * cfg.epochs
    warmup = max(1, int(steps * cfg.warmup_frac))
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda s: (s + 1) / warmup if s < warmup
        else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, steps - warmup))))

    loss_fn = nn.BCEWithLogitsLoss()      # soft targets are fine: BCE takes [0,1]
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and cfg.device == "cuda")

    best = {"auc": -np.inf}
    best_state, best_preds, history, stale = None, None, [], 0

    for epoch in range(cfg.epochs):
        t0 = time.time()
        train_loss, _, _ = run_epoch(model, train_loader, loss_fn, cfg,
                                     optimizer, scheduler, scaler)
        val_loss, val_prob, val_true = run_epoch(model, val_loader, loss_fn, cfg)
        metrics = multilabel_metrics(val_true, val_prob, labels)

        history.append({"fold": fold, "epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "val_auc": metrics["auc"],
                        "val_ap": metrics["ap"], "seconds": time.time() - t0})
        if verbose:
            print(f"  fold {fold} epoch {epoch}  train {train_loss:.4f}  "
                  f"val {val_loss:.4f}  auc {metrics['auc']:.3f}  "
                  f"ap {metrics['ap']:.3f}  ({time.time() - t0:.0f}s)")

        # `best_state is None` first: a fold whose validation set has no label
        # with both classes scores NaN, and without this it would never save a
        # checkpoint and would fall over building the out-of-fold frame.
        score = metrics["auc"]
        improved = best_state is None or (not np.isnan(score) and score > best["auc"])
        if improved:
            best, stale = metrics, 0
            best_preds = val_prob
            # .cpu() on the way out: a deepcopy of a CUDA state_dict puts a
            # second full set of weights on the GPU, next to the live model.
            best_state = {k: v.detach().to("cpu", copy=True)
                          for k, v in model.state_dict().items()}
            torch.save(best_state, ckpt_path)
        else:
            stale += 1
            if stale >= cfg.patience:
                if verbose:
                    print(f"  fold {fold}: no gain in {cfg.patience} epochs, stopping")
                break

    oof = val_df[["study", "series"]].copy()
    for i, lab in enumerate(labels):
        oof[f"pred_{lab}"] = best_preds[:, i]
        oof[f"true_{lab}"] = val_df[lab].values
    oof["fold"] = fold

    # Hand back the checkpoint path, not the model: five live fold models plus
    # their optimiser state is most of a GPU, and every one of them is already
    # on disk.
    del model
    free_memory()
    return ckpt_path, pd.DataFrame(history), best, oof


def cross_validate(df, labels=None, cfg=None, folds=None, verbose=True):
    """Train every fold. Returns (checkpoints, oof, history, summary).

    `checkpoints` maps fold -> path on disk. Pass it straight to predict(),
    which loads them one at a time.

    The out-of-fold frame is the useful artefact: every training study has a
    prediction from a model that never saw it, so thresholds and ensembling can
    be tuned on it without touching the test split.
    """
    labels = list(labels if labels is not None else LABELS)
    cfg = cfg or TrainConfig()
    folds = folds if folds is not None else sorted(
        df.loc[df["split"] == "train", "fold"].unique())

    models, oofs, histories, rows = {}, [], [], []
    for fold in folds:
        if verbose:
            print(f"\n=== fold {fold} ===")
        ckpt, history, best, oof = train_fold(df, fold, labels, cfg, verbose)
        models[fold] = ckpt                     # a path on disk, not a module
        oofs.append(oof)
        histories.append(history)
        rows.append({"fold": fold, "val_auc": best["auc"], "val_ap": best["ap"],
                     "skipped_labels": len(best["skipped"])})

    oof = pd.concat(oofs, ignore_index=True)
    summary = pd.DataFrame(rows)
    if verbose and len(summary):
        print(f"\nCV AUC {summary['val_auc'].mean():.3f} "
              f"+/- {summary['val_auc'].std():.3f}")
    return models, oof, pd.concat(histories, ignore_index=True), summary


@torch.no_grad()
def predict(models, df, labels, cfg, verbose=True):
    """Average the fold models over a held-out frame. Returns predictions + metrics.

    Touch the test split once, at the end. Every look at it that changes a
    decision turns it into a second validation set.
    """
    loader = make_loader(df, labels, cfg, train=False)

    # One fold on the device at a time, averaged as we go: holding five
    # models to average at the end costs five times the weights for no reason.
    mean_prob, n_models = None, 0
    for fold, model in models.items():
        if isinstance(model, str):              # a checkpoint path
            module = DinoV2MultiLabel(len(labels), cfg)
            module.load_state_dict(torch.load(model, map_location="cpu"))
        else:
            module = model
        module.eval().to(cfg.device)

        probs = []
        for x, _ in loader:
            logits = module(x.to(cfg.device, non_blocking=True))
            probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        probs = np.concatenate(probs)
        mean_prob = probs if mean_prob is None else mean_prob + probs
        n_models += 1

        module.to("cpu")
        del module
        free_memory()
    mean_prob /= max(n_models, 1)
    y_true = df[labels].to_numpy(dtype=np.float32)
    metrics = multilabel_metrics(y_true, mean_prob, labels)

    out = df[["study", "series"]].copy()
    for i, lab in enumerate(labels):
        out[f"pred_{lab}"] = mean_prob[:, i]
    if verbose:
        print(f"test AUC {metrics['auc']:.3f}  AP {metrics['ap']:.3f}  "
              f"({n_models} models, {len(df)} series)")
    return out, metrics

In [ ]:
# ============================================================================
# Train
# ============================================================================
# For training, leave the pixels on disk: the Dataset reads only the slices it
# samples, per batch. `build_dataset` will refuse to fill the `image` column
# past memory_budget_gb anyway and tell you what it would have cost.
df = build_dataset(DATA_PATH, load_images=False)
df = make_splits(df, n_folds=5, test_frac=0.15, seed=42)

cfg = TrainConfig(epochs=8, n_slices=16, batch_size=4)
gpu_report("before: ")

# Returns a checkpoint path per fold, not five live models.
ckpts, oof, history, summary = cross_validate(df, LABELS, cfg)
gpu_report("after:  ")
summary

In [ ]:
# Per-label out-of-fold AUC: where the model is actually working.
from sklearn.metrics import roc_auc_score

for lab in LABELS:
    true = (oof[f"true_{lab}"] > 0.5).astype(int)
    if true.nunique() < 2:
        print(f"{lab:20s}   no positives out of fold")
        continue
    print(f"{lab:20s} {roc_auc_score(true, oof[f'pred_{lab}']):.3f}  "
          f"({true.sum()} positive of {len(true)})")

# The test split, once, at the end. predict() loads one fold at a time.
test_df = df[df["split"] == "test"].reset_index(drop=True)
test_preds, test_metrics = predict(ckpts, test_df, LABELS, cfg)

In [ ]:
# What the attention pool looked at. A model weighting the mid-stack slices is
# looking at the joint; one weighting the first and last is looking at the edge
# of the field of view, which usually means the head learned an artefact.
import matplotlib.pyplot as plt

model = DinoV2MultiLabel(len(LABELS), cfg)
model.load_state_dict(torch.load(ckpts[0], map_location="cpu"))
model.eval().to(cfg.device)

x, y = KneeVolumeDataset(test_df.head(4), LABELS, cfg.n_slices, cfg.image_size)[0]
with torch.no_grad():
    _, weights = model(x.unsqueeze(0).to(cfg.device), return_weights=True)

plt.figure(figsize=(6, 2.5))
plt.bar(range(cfg.n_slices), weights[0].cpu().numpy())
plt.xlabel("slice (through-plane order)"); plt.ylabel("attention")
plt.tight_layout(); plt.show()

model.to("cpu"); del model; free_memory()